# Домашнее задание (50 баллов)

В этом домашнем задании вы познакомитесь с основами NLP, научитесь обрабатывать тексты.

В местах, где используется `...` (elipsis), требуется заменить его на код.

Установим необходимые зависимости:

In [ ]:
!pip install -U pip
!pip install nltk tqdm seqeval scikit-learn datasets numpy

In [1]:
from typing import List, Dict, Tuple, Callable, Optional

## Токенизация (15 баллов)

Токенизация - это процесс преобразования текста в набор токенов.
Наивная реализация разбивает текст по пробелам. Более умные реализации учитывают пунктуацию.

### Библиотека NLTK (2 балла)

Научимся работать с токенизацией NLTK, где уже [реализована](https://www.nltk.org/api/nltk.tokenize.html#nltk.tokenize.word_tokenize) работа с пунктуацией.

https://www.nltk.org/

In [2]:
import nltk


# https://www.nltk.org/nltk_data/
# nltk.download("punkt")
# nltk.download('punkt_tab')


def tokenize(text: str, language: str = "english") -> List[str]:
    
    tokenized = nltk.word_tokenize(
        text=text,
        language=language
    )
    
    return tokenized
    
assert tokenize("") == []
assert tokenize("Hello, world!") == ["Hello", ",", "world", "!"]
assert tokenize("EU rejects German call to boycott British lamb.") == ["EU", "rejects", "German", "call", "to", "boycott", "British", "lamb", "."]

### Нормализация (3 балла)

Добавим нормализацию после токенизации. Пробуем [лемматизацию](https://www.nltk.org/api/nltk.stem.wordnet.html#nltk.stem.wordnet.WordNetLemmatizer) , [стемминг](https://www.nltk.org/api/nltk.stem.snowball.html#nltk.stem.snowball.EnglishStemmer) и [юникод](https://docs.python.org/3/library/unicodedata.html#unicodedata.normalize) нормализацию. Напишем функцию, которая будет принимать на вход токен после токенизации, нормализовать в NFC юникод форму, переводит в нижний регистр, лемматизирует слово и, если слово не изменилось после лемматизации, применяет стемминг.


Создайте функцию `normalize`:
   - Функция `normalize` должна принимать строку `token` и возвращать нормализованный токен.
   - Примените к токену Unicode нормализацию с помощью `unicode_nfc_normalizer`.
   - Преобразуйте токен в нижний регистр.
   - Примените лемматизацию с помощью `lemmatizer`.
   - Если лемматизированный токен отличается от исходного, верните его. В противном случае, примените стемминг с помощью `stemmer` и верните результат.

In [3]:
import unicodedata

from nltk.stem.snowball import EnglishStemmer
from nltk.stem.wordnet import WordNetLemmatizer


# nltk.download('wordnet')

stemmer = EnglishStemmer()
lemmatizer = WordNetLemmatizer()


def normalize(token: str) -> str:
    """
    Нормализует токен, применяя Unicode нормализацию, преобразование в нижний регистр,
    лемматизацию и стемминг при необходимости.

    :param token: Токен для нормализации
    :return: Нормализованный токен
    """
    
    if not token:
        return ""
    
    token = unicodedata.normalize('NFC', token)
    token = token.lower()
    token = lemmatizer.lemmatize(token)
    token = stemmer.stem(token)
    
    return token


test_tokens = ["Worlds", "churches", "Helping"]
assert [normalize(token) for token in test_tokens] == ["world", "church", "help"]

### Добавляем Словарь (10 баллов)

Современные токенайзеры не только разбивают строки на токены, но и преобразуют последовательность токенов в последовательность числел. Объединим функцию токенизации, нормализации и отображения из токенов в индексы в один объект токенайзера.

Напишите класс `Tokenizer` для токенизации и нормализации текста.

Построение словаря:
   - Создайте метод `_build_vocabulary`, который принимает список текстов `texts` и обновляет словарь токенов.
   - Для каждого текста:
     - Токенизируйте и нормализуйте текст.
     - Обновите счетчик вхождений слов.
   - Для каждого слова, которое встречается не менее `min_count` раз, добавьте слово в словарь `word2idx` и список `idx2word`.

Кодирование и декодирование:
   - Создайте метод `encode_word`, который принимает слово `word` и возвращает его индекс с применением нормализации.
   - Создайте метод `encode`, который принимает текст `text` и возвращает список индексов токенов.
   - Создайте метод `decode`, который принимает список индексов `input_ids` и возвращает текст, вставляя пробелы между токенами.

> Note: для функций, которые могут долго исполнятся (`_build_vocab`), рекомендуется использовать библиотеку tqdm.

In [4]:
from collections import Counter
from tqdm import tqdm_notebook as tqdm


class Tokenizer:
    def __init__(
            self, 
            texts: List[str], 
            tokenize_fn: Callable[[str], List[str]] = tokenize,
            normalize_fn: Callable[[str], str] = lambda token: token,
            min_count: int = 1,
    ) -> None:
        """
        Инициализация токенизатора.

        :param texts: список текстов для построения словаря
        :param tokenize_fn: функция для токенизации текста
        :param normalize_fn: функция для нормализации токенов
        :param min_count: минимальное количество вхождений слова для включения в словарь
        """
        self.min_count = min_count
        self.tokenize = tokenize_fn
        self.normalize = normalize_fn
        self.word2idx = {"<PAD>": 0, "<BOS>": 1, "<EOS>": 2, "<UNK>": 3}
        self.unk_token_id = 3
        self.idx2word = ["<PAD>", "<BOS>", "<EOS>", "<UNK>"]
        self.word2count = Counter()
        self._build_vocabulary(texts)

    def _build_vocabulary(self, texts: List[str]):
        """
        Построение словаря на основе списка текстов.
        """
        all_tokens = []
        
        for text in tqdm(texts, desc="Building vocabulary"):
            tokens = self.tokenize(text)
            normalized_tokens = [self.normalize(token) for token in tokens]
            all_tokens.extend(normalized_tokens)
        
        self.word2count.update(all_tokens)
        
        for word, count in self.word2count.items():
            if count >= self.min_count and word not in self.word2idx:
                idx = len(self.word2idx)
                self.word2idx[word] = idx
                self.idx2word.append(word)
        
        
    def encode_word(self, word: str) -> int:
        """
        Кодирование слова в индекс с применением нормализации.
        """
        normalized_word = self.normalize(word)
        return self.word2idx.get(normalized_word, self.unk_token_id)

    def encode(self, text: str) -> List[int]:
        """
        Кодирование текста в набор индексов.
        """
        tokens = self.tokenize(text)
        return [self.encode_word(token) for token in tokens]

    def decode(self, input_ids: List[int]) -> str:
        """
        Декодирование набора индексов в текст. Вставляет пробел между декодированнми токенами.

        :param input_ids: набор индексов токенов
        :return: текст
        """
        return " ".join([self.idx2word[token] for token in input_ids])

    def __len__(self) -> int:
        """
        Возвращает количество уникальных токенов в словаре.

        :return: количество уникальных токенов
        """
        return len(self.word2idx)

    def __contains__(self, item: str) -> bool:
        """
        Проверяет, содержится ли слово в словаре.

        :param item: слово
        :return: True, если слово содержится в словаре, иначе False
        """
        return item in self.word2idx

    def __str__(self):
        """
        Возвращает строковое представление словаря.

        :return: строковое представление словаря
        """
        return str(self.word2idx)

In [5]:
corpus = ["Hello, world!", "I love Python!"]

tokenizer = Tokenizer(corpus, min_count=1)
encoded = tokenizer.encode("Hello, Python! I love you")
assert tokenizer.decode(encoded) == "Hello , Python ! I love <UNK>"

tokenizer = Tokenizer(corpus, normalize_fn=normalize)
encoded = tokenizer.encode("Hello, Python! I loved you")
assert tokenizer.decode(encoded) == "hello , python ! i love <UNK>"

C:\Users\germa\AppData\Local\Temp\ipykernel_7416\2351445109.py:36: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for text in tqdm(texts, desc="Building vocabulary"):


Building vocabulary:   0%|          | 0/2 [00:00<?, ?it/s]

Building vocabulary:   0%|          | 0/2 [00:00<?, ?it/s]

## TF-IDF (20 баллов)


### Класс TFIDF (10 баллов)

Создайте класс `TFIDF` для вычисления TF-IDF значений. Формулы для подсчёта TF и IDF можно выбрать [тут](https://en.wikipedia.org/wiki/Tf%E2%80%93idf).

Обучение модели должно осуществляться с помощью метода `fit`, который принимает список строк `docs` и обучает модель на этом корпусе, вызывая метод `add_doc` для каждого документа.

Предсказание TF-IDF значений:
   - Создайте метод `predict`, который принимает список строк `docs` и возвращает матрицу TF-IDF значений.
   - Для каждого документа:
     - Токенизируйте документ.
     - Вычислите TF для каждого термина.
     - Вычислите IDF для каждого термина.
     - Заполните матрицу TF-IDF значений.
   - Нормализуйте строки матрицы, чтобы сумма значений в каждой строке была равна 1.

> Важно! Не забудьте убрать `<UNK>` токен во  время подсчёта TF-IDF 

Для функций, которые могут долго исполнятся (`fit`, `predict`), рекомендуется использовать библиотеку tqdm.

In [6]:
from collections import Counter
from typing import List
import numpy as np


class TFIDF:
    def __init__(self, tokenizer: Tokenizer, default_idf = 1.0) -> None:
        """
        Инициализация TFIDF.

        :param tokenizer: токенизатор для преобразования текста в токены
        :param default_idf: значение IDF для неизвестных токенов
        """
        self.tokenizer = tokenizer
        self.num_docs = 0
        self.term2num_docs = [0 for _ in tokenizer.word2idx]  # для подсчёта IDF
        self.default_idf = default_idf

    @property
    def vocab_size(self) -> int:
        """
        Возвращает размер словаря.

        :return: размер словаря
        """
        return len(self.tokenizer)

    def add_doc(self, doc: str) -> None:
        """
        Добавляет документ в модель TFIDF.

        :param doc: документ для добавления
        """
        
        tokenized = self.tokenizer.encode(doc)
        self.num_docs += 1
        
        unique_indices = set(tokenized)
        for idx in unique_indices:
            if idx < len(self.term2num_docs):
                self.term2num_docs[idx] += 1

    def fit(self, docs: List[str]) -> None:
        """
        Обучает модель TFIDF на корпусе docs.

        :param docs: корпус для обучения
        """
        
        for doc in docs:
            self.add_doc(doc)

    def predict(self, docs: List[str]) -> np.ndarray:
        """
        Предсказывает TFIDF значения для списка документов.

        :param docs: список документов
        :return: матрица TFIDF значений
        """
        
        output_matrix = np.zeros((len(docs), self.vocab_size))
        for idx, doc in enumerate(docs):
            
            tokenized_doc = self.tokenizer.encode(doc)
            words_frequency = Counter(tokenized_doc)
            
            tfidf_vector = []
            
            for term_idx in range(self.vocab_size):
                
                if term_idx == self.tokenizer.unk_token_id:
                    tfidf_vector.append(0)
                    continue
                
                tf = words_frequency.get(term_idx, 0)
                idf_val = self.idf(term_idx)
                tfidf_vector.append(tf * idf_val)
                
            output_matrix[idx] = tfidf_vector
            
        return output_matrix
                

    def idf(self, term: int) -> float:
        """
        Вычисляет IDF (обратную частоту документа) для термина.

        :param term: термин
        :return: IDF значение
        """
        
        if term < 0 or term >= len(self.term2num_docs):
            return self.default_idf
        
        frequency = self.term2num_docs[term]
        if self.num_docs == 0:
            return self.default_idf
        
        smoothed = np.log(self.num_docs / (frequency + 1)) + 1
        return smoothed
    
        # тестовый корпус из ячейки ниже содержит всего 2 текста
        # токен "test" появляется в обоих и в классической формуле
        # idf("test") == log(2 / 2) == 0
        # поэтому в домашнем задании рекомендуется использовать
        # сглаженный вариант подсчёта inverse document frequency smooth:
        # log(N / (n + 1)) + 1
        # https://en.wikipedia.org/wiki/Tf%E2%80%93idf#Inverse_document_frequency
        ...

Тесты были проверены для такой комбинации формул:

$$
\text{TF} = \frac{f_{t,d}}{\sum_{t' \in d} f_{t',d}}
$$
$$
\text{IDF} = \log{\frac{N}{1 + n_t}} + 1
$$

Если вы выбрали другие формулы для подсчёта, то можно поправить тесты соответственно.

> Можно расширить расширить класс для удобства и поэксперементировать с различными формулами для подсчёта TF и IDF при классификации IMDB датасета ниже.

In [7]:
corpus = ["test test", "not a test"]
tokenizer = Tokenizer(corpus)
tfidf = TFIDF(tokenizer)
tfidf.fit(corpus)

assert tfidf.vocab_size == 4 + 3
# 3 токена, один из которых <UNK>
vector = tfidf.predict(["a test string"])[0]
# tf("a") == tf("test") and idf("a") > idf("test")
assert vector[tfidf.tokenizer.word2idx["a"]] > vector[tfidf.tokenizer.word2idx["test"]]

vector = tfidf.predict(["not a test a string"])[0]
assert vector[tfidf.tokenizer.word2idx["a"]] > 2 * vector[tfidf.tokenizer.word2idx["test"]]
assert vector[tfidf.tokenizer.word2idx["a"]] == 2 * vector[tfidf.tokenizer.word2idx["not"]]

assert not np.any(tfidf.predict(["all tokens abscent from vocab should be zeros vector"]))

C:\Users\germa\AppData\Local\Temp\ipykernel_7416\2351445109.py:36: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for text in tqdm(texts, desc="Building vocabulary"):


Building vocabulary:   0%|          | 0/2 [00:00<?, ?it/s]

### Датасет (1 балл)

В качестве датасета вёзмём популярный [набор отзывов на фильмы с сайта IMDB](https://huggingface.co/datasets/stanfordnlp/imdb). Нужно предсказать является ли отзыв позитивным или негативным. Чтобы скачать датасет воспользуемся библиотекой `datasets` из экосистемы `HuggingFace`. Интерфейс датасета похож на словарь, доступ к разным частям осуществляется по названию ключа:

1. Тренировочная часть датасета: `imdb["train"]`
2. Тексты для тренировки: `imdb["train"]["text"]`
3. Лейблы для тренировки: `imdb["train"]["label"]`

In [9]:
from datasets import load_dataset


imdb = load_dataset("stanfordnlp/imdb")

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

### Тренируем Токенайзер (2 балла)

Используя `"train"` часть датасета, инициализируйте два токеназйера:
1. Не использующий нормализацию
2. Использующий функцию `normalize`, определённую выше

Сравним размер полученного словаря в обоих случаях.

In [22]:
tokenizer_without_norm = Tokenizer(imdb["train"]["text"])
tokenizer_with_norm = Tokenizer(imdb["train"]["text"], normalize_fn=normalize)

assert len(tokenizer_without_norm) > len(tokenizer_with_norm)

C:\Users\germa\AppData\Local\Temp\ipykernel_7416\2351445109.py:36: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for text in tqdm(texts, desc="Building vocabulary"):


Building vocabulary:   0%|          | 0/25000 [00:00<?, ?it/s]

Building vocabulary:   0%|          | 0/25000 [00:00<?, ?it/s]

### Тренируем TF-IDF Модель (2 балла)

Теперь мы можем натренировать модель на датасете. Обучим две модели, которые будут использовать разные токенайзеры.

In [23]:
tfidf_with_norm = TFIDF(tokenizer_with_norm)

In [24]:
tfidf_without_norm = TFIDF(tokenizer_without_norm)

### Обучим Логистическую Регрессию (5 баллов)

В качестве входов в модель нужно использовать TF-IDF представления документов (`X_train`), в качестве лейблов - 0 и 1, обозначающие нужный класс (`Y_train`). Начнём с модели, которая использует нормализацию.

Используюя тестовый датасет и `logreg.predict` проверьте предсказания модели, вычислив accuracy - количество правильных предсказаний, делённое на количество входных примеров.

In [33]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

n_samples = 1000

X_train_text, _, Y_train, _ = train_test_split(
    imdb["train"]["text"],
    imdb["train"]["label"],
    train_size=n_samples,
    stratify=imdb["train"]["label"],
    random_state=42
)
X_train = tfidf_with_norm.predict(X_train_text)

logreg = LogisticRegression()
logreg.fit(X_train, Y_train) 

n_test_samples = 200
X_test_text, _, Y_test, _ = train_test_split(
    imdb["test"]["text"],
    imdb["test"]["label"],
    train_size=n_test_samples,
    stratify=imdb["test"]["label"],
    random_state=42
)
X_test = tfidf_with_norm.predict(X_test_text)

preds = logreg.predict(X_test)
accuracy = (preds == Y_test).sum() / n_test_samples
print("Accuracy: ", accuracy)

c:\Users\germa\AppData\Local\miniconda3\envs\work_env\lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Accuracy:  0.755


Теперь обучим логистическую регрессию со второй TF-IDF моделью и сравним результаты:

In [34]:
n_samples = 1000
X_train_text, _, Y_train, _ = train_test_split(
    imdb["train"]["text"],
    imdb["train"]["label"],
    train_size=n_samples,
    stratify=imdb["train"]["label"],
    random_state=42
)
X_train = tfidf_without_norm.predict(X_train_text)

logreg_without_norm = LogisticRegression()
logreg_without_norm.fit(X_train, Y_train) 

n_test_samples = 200
X_test_text, _, Y_test, _ = train_test_split(
    imdb["test"]["text"],
    imdb["test"]["label"],
    train_size=n_test_samples,
    stratify=imdb["test"]["label"],
    random_state=42
)
X_test = tfidf_without_norm.predict(X_test_text)

preds = logreg_without_norm.predict(X_test)
accuracy = (preds == Y_test).sum() / n_test_samples
print("Accuracy: ", accuracy)

c:\Users\germa\AppData\Local\miniconda3\envs\work_env\lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Accuracy:  0.76


### (Опционально) TfidfVectorizer

Можете изучть класс [TfidfVectorizer](https://scikit-learn.org/1.5/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html) из библиотеки scikit-learn и сравнить его со своей имплементацией, обучив логистическую регрессию с его помощью.

## $n$-граммные языковые модели (15 баллов)

### Расширяем Токенайзер (3 балла)

Перед созданием языковой модели, расширим токенизационный класс. Добавим два флага в сигнатуру метода `encode`, чтобы управлять добавлением служебных токенов во время токенизации. Существующий метод `decode` уже пропускает `<PAD>` токен, добавим флаг `skip_special_tokens` для пропуска всех специальных токенов.

In [35]:
class BoSTokenizerEoS(Tokenizer):
    
    def __init__(self, texts: List[str], tokenize_fn: Callable = tokenize, 
                 normalize_fn: Callable = lambda token: token, min_count: int = 1):
        super().__init__(texts, tokenize_fn, normalize_fn, min_count)
        self.special_tokens = {"<PAD>", "<BOS>", "<EOS>", "<UNK>"}
    
    def encode(self, text: str, add_bos: bool = True, add_eos: bool = False) -> List[int]:
        """
        Кодирование текста в набор индексов.

        :param text: текст
        :param add_bos: добавление begin-of-sentence токена в начало
        :param add_eos: добавление end-of-sentence токена в конец
        :return: набор индексов токенов
        """
        
        encoded = []
        if add_bos:
            encoded.append(self.word2idx.get("<BOS>", self.unk_token_id))
        
        tokens = self.tokenize(text)
        encoded.extend([self.encode_word(token) for token in tokens])
        
        if add_eos:
            encoded.append(self.word2idx.get("<EOS>", self.unk_token_id))
            
        return encoded

    def decode(self, input_ids: List[int], skip_special_tokens: bool = True) -> str:
        """
        Декодирование набора индексов в текст. Вставляет пробел между декодированнми токенами.

        :param input_ids: набор индексов токенов
        :param skip_special_tokens: пропуск специальных токенов во время декодирования.
        :return: текст
        """
        
        tokens = []
        for token_id in input_ids:
            # Проверяем, является ли токен специальным
            token = self.idx2word[token_id] if token_id < len(self.idx2word) else "<UNK>"
            
            if skip_special_tokens:
                # Пропускаем, если токен в множестве специальных
                if token in self.special_tokens:
                    continue
            
            tokens.append(token)
        
        return " ".join(tokens)

### Создаём NGram Модель (12 баллов)

Создайте класс `NGramLanguageModel` для построения n-граммной языковой модели. В этом задании вы можете как опираться на предложенную структуру модели, так и сделать свою имплементацию.

Построение модели:
   - Создайте метод `_build_model`, который принимает список текстов `texts` и обновляет частоты n-грамм.
   - Для каждого текста:
     - Токенизируйте текст и добавьте токен `"<EOS>"` в конец.
     - Для каждого токена:
       - Определите префикс длиной `n-1`.
       - Обновите частоты n-грамм и частоты префиксов.

Генерация следующего токена:
   - Создайте метод `generate_next_token`, который принимает префикс `prefix` и возвращает следующий токен.
   - Преобразуйте префикс в кортеж.
   - Получите распределение частот для префикса.
   - Если распределение пустое, верните токен `"<UNK>"`.
   - Верните токен с наибольшей частотой.

Автодополнение текста:
   - Создайте метод `autocomplete`, который принимает текст `text` и максимальную длину `max_len`, и возвращает завершенный текст.
   - Токенизируйте текст.
   - Пока длина токенов меньше `max_len`:
     - Определите префикс длиной `n-1`.
     - Сгенерируйте следующий токен.
     - Добавьте токен в список токенов.
     - Если токен равен `"<EOS>"`, завершите генерацию.
   - Декодируйте и верните текст.

In [53]:
from collections import defaultdict


class NGramLanguageModel:
    def __init__(self, n: int, tokenizer: BoSTokenizerEoS, texts: List[str]):
        """
        Создание n-граммной языковой модели.

        :param n: порядок n-грамм
        :param vocabulary: словарь
        """
        assert n >= 2
        self.n = n
        self.tokenizer = tokenizer
        self.frequencies = defaultdict(lambda: Counter())  # частота n-грамм
        self.frequencies_of_prefixes = Counter()  # сумма частот n-грамм для префиксов
        self._build_model(texts)

    def _build_model(self, texts: List[str]):
        """
        Построение модели на основе списка текстов.

        :param texts: список текстов
        """
        
        for text in texts:
            
            tokens = self.tokenizer.encode(
                text=text,
                add_bos=True,
                add_eos=True
            )
            
            for i in range(len(tokens) - self.n + 1):
                
                prefix = tuple(tokens[i:i + self.n - 1])
                next_token = tokens[i + self.n - 1]
                
                self.frequencies[prefix][next_token] += 1
                self.frequencies_of_prefixes[prefix] += 1

    def generate_next_token(self, prefix: List[int]) -> int:
        """
        Жадная генерация следующего токена по префиксу.

        :param prefix: префикс
        :return: следующий токен
        """
        
        prefix_tuple = tuple(prefix)
        next_token_distribution = self.frequencies.get(prefix_tuple)
        
        if not next_token_distribution:
            return self.tokenizer.word2idx.get("<UNK>", 3)
        
        most_common_token = max(next_token_distribution, key=next_token_distribution.get)
        return most_common_token

    def autocomplete(self, text: str, max_len: int = 32, skip_special_tokens: bool = True) -> str:
        """
        Автоматическое дополнение текста.

        :param text: текст
        :param max_len: максимальная длина текста
        :param skip_special_tokens: пропуск специальных токенов во время декодирования.
        :return: завершенный текст
        """
        
        tokens = self.tokenizer.encode(text)
        
        while len(tokens) < max_len:
            
            prefix = tokens[-(self.n - 1):]
            
            if len(prefix) < self.n - 1:
                prefix = [self.tokenizer.word2idx["<BOS>"]] * (self.n - 1 - len(prefix)) + prefix
            
            next_token = self.generate_next_token(prefix)
            
            idx_eos = self.tokenizer.word2idx.get("<EOS>", 2)
            if next_token == idx_eos:
                tokens.append(idx_eos)
                break
            
            tokens.append(next_token)
            
        return self.tokenizer.decode(tokens, skip_special_tokens=skip_special_tokens)

In [54]:
corpus = ["Hello, world!", "I love Python!", "Hello, Python"]
tokenizer = BoSTokenizerEoS(corpus, min_count=1)
ngram_lm = NGramLanguageModel(2, tokenizer, corpus)

print(ngram_lm.autocomplete("Hello, Python", max_len=10, skip_special_tokens=False))
assert ngram_lm.autocomplete("Hello, Python", max_len=10) == "Hello , Python !"
assert ngram_lm.autocomplete("Hello, Python", max_len=10, skip_special_tokens=False) == "<BOS> Hello , Python ! <EOS>"

C:\Users\germa\AppData\Local\Temp\ipykernel_7416\2351445109.py:36: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for text in tqdm(texts, desc="Building vocabulary"):


Building vocabulary:   0%|          | 0/3 [00:00<?, ?it/s]

<BOS> Hello , Python ! <EOS>


# Комментарии

Если остались вопросы, на которые хочется получить ответ при ревью, это место для них: